# Airspeed velocity demo

In [ ]:
%%writefile asv.conf.json
{
  "version": 1,
  "project": "asv_ysda_demo",
  "repo": ".",
  "environment_type": "existing",
  "benchmark_dir": "benchmarks",
  "build_command": [],
  "matrix": {},
  "results_dir": "results"
}


In [ ]:
%%writefile benchmarks/bench_ops.py
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


import torch
from mypackage.silu import get_silu

def check_cuda_avail():
    if not torch.cuda.is_available():
        raise NotImplementedError(
            "Torch could not detect your GPU, so..."
        )


class TimeFusionCUDA:
    params = [1024, 4096]
    param_names = ["size"]

    def setup(self, size):
        check_cuda_avail()

        self.x = torch.randn(size, size, device="cuda")
        # compile if necessary
        self.silu = get_silu()
        # warmup (avoid measuring compile)
        self.silu(self.x)
        torch.cuda.synchronize()

    def time_unfused(self, size):
        self.silu(self.x)
        torch.cuda.synchronize()


class MemFusionCUDA:
    params = [1024, 4096]
    param_names = ["size"]

    def setup(self, size):
        check_cuda_avail()
        
        self.x = torch.randn(size, size, device="cuda")
        self.silu = get_silu()
        self.silu(self.x)
        torch.cuda.synchronize()

    def peakmem_unfused(self, size):
        self.silu(self.x)
        torch.cuda.synchronize()


In [ ]:
%%writefile mypackage/silu.py
import torch
import typing as tp

def get_silu() -> tp.Callable[[torch.Tensor], torch.Tensor]:
    def fn(x):
        return x * x.sigmoid()
    return fn

In [ ]:
%%bash

git add src
git commit -am 'Silly silu'

In [ ]:
%%bash
uv run asv run --python=same --set-commit-hash=$(git rev-parse HEAD)

In [ ]:
%%writefile src/mypackage/silu.py
import torch
import typing as tp

def get_silu() -> tp.Callable[[torch.Tensor], torch.Tensor]:
    def fn(x):
        return x * x.sigmoid()

    return torch.compile(fn)

In [ ]:
%%bash

git add mypackage
git commit -am 'Compiled silu'

In [ ]:
%%bash
uv run asv run --python=same --set-commit-hash=$(git rev-parse HEAD)

In [ ]:
%%bash

uv run asv publish
uv run asv preview